# 🔍 Relational Fraud Detection
### Graph Neural Network on Heterogeneous Transaction Graphs

---

**Dataset:** IEEE-CIS Fraud Detection (Kaggle)  
**Task:** Binary Node Classification — Fraud vs. Legitimate  
**Target:** `isFraud` (0 = Legitimate, 1 = Fraud)  
**Framework:** PyTorch — implemented from scratch


---
## Phase 01 — Data Preprocessing

**Goal:** Load the raw data, clean it, and prepare it so we can build a graph from it in Phase 03.

The IEEE-CIS dataset comes in two separate files — one for transactions and one for identity/device info. Our job here is to merge them, handle missing values, encode categories, and identify which columns will later become nodes in our graph.

### 1.1 — Imports

Load all libraries we need across this phase.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded.')

### 1.2 — Load Data

The dataset has two files:
- `train_transaction.csv` — 394 columns. Contains the transaction itself: amount, card IDs, product codes, addresses, email domains, and 339 Vesta-engineered V-features.
- `train_identity.csv` — 41 columns. Contains device/browser info linked to a transaction via `TransactionID`.

Not every transaction has an identity record — only about 40% do. That missing identity data is actually a signal: bots and scripted fraud often have no device metadata at all.

In [ ]:
train_trans = pd.read_csv('/content/sample_data/train_transaction.csv')
train_id    = pd.read_csv('/content/sample_data/train_identity.csv')

print(f'Transactions : {train_trans.shape[0]:,} rows x {train_trans.shape[1]} cols')
print(f'Identity     : {train_id.shape[0]:,} rows x {train_id.shape[1]} cols')
print(f'Coverage     : {train_id["TransactionID"].nunique() / train_trans.shape[0] * 100:.1f}% of transactions have identity data')

### 1.3 — Merge

We join the two tables on `TransactionID` using a **left join** — this keeps all transactions and attaches identity data where it exists, filling `NaN` elsewhere.

In [ ]:
df = pd.merge(train_trans, train_id, on='TransactionID', how='left')

print(f'Merged shape : {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Fraud count  : {df["isFraud"].sum():,}  ({df["isFraud"].mean() * 100:.2f}% of transactions)')

df.head(3)

### 1.4 — Class Distribution

The dataset is heavily imbalanced — roughly 3.5% of transactions are fraud. This is realistic for financial data but it means we cannot use accuracy as a metric. A model that always predicts "not fraud" would get 96.5% accuracy while being completely useless.

We will handle this in Phase 05 with weighted loss functions. For now we just confirm the ratio.

In [ ]:
counts = df['isFraud'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'crimson'], width=0.5)
ax.set_title('Class Distribution — isFraud')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 1000, f'{v:,}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print(f'Ratio: 1 fraud per {int(counts[0] / counts[1])} legitimate transactions')
print('Note: Accuracy is useless here. We will use AUC-ROC and F1 in Phase 06.')

### 1.5 — Missing Value Audit

Before fixing anything, we understand the scale of the problem. We count nulls per column and visualize the worst offenders.

Key observation: identity columns (`id_*`, `DeviceType`, `DeviceInfo`) are the most sparse — expected since only 40% of transactions have identity records at all.

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

print(f'Columns with missing values : {len(null_pct)} / {df.shape[1]}')
print(f'Columns fully complete      : {df.shape[1] - len(null_pct)}')

fig, ax = plt.subplots(figsize=(12, 6))
null_pct.head(30).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.axvline(50, color='crimson', linestyle='--', linewidth=1, label='50% missing')
ax.set_xlabel('Missing (%)')
ax.set_title('Top 30 Columns by Missing Rate')
ax.legend()
plt.tight_layout()
plt.show()

### 1.6 — Imputation

We use a type-aware strategy:
- **Categorical** → fill with `'unknown'`. Keeps missingness as its own category rather than pretending the data exists.
- **Numerical** → fill with the **column median**. We use median instead of mean because financial data is right-skewed — a few huge transactions would make the mean a bad fill value.

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = [c for c in df.select_dtypes(include=['int64', 'float64']).columns if c != 'isFraud']

df[cat_cols] = df[cat_cols].fillna('unknown')

for col in num_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print(f'Categorical columns filled : {len(cat_cols)}')
print(f'Remaining nulls            : {df.isnull().sum().sum()}')

### 1.7 — Label Encoding

GNNs work with numbers. We label-encode all categorical columns so every value in the dataframe is numeric.

Note: columns like `card1`, `DeviceInfo`, and `P_emaildomain` will not be used as flat features — they become **entity nodes** in the graph in Phase 03. We encode them now just to keep the dataframe clean and consistent.

In [ ]:
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

print(f'Encoded {len(cat_cols)} categorical columns.')
print(f'Remaining object columns: {(df.dtypes == "object").sum()}  (should be 0)')

### 1.8 — Normalization

We apply `StandardScaler` to all float columns — centers them at 0 with unit variance. GNNs aggregate neighbor features during message passing, so large feature scales cause unstable gradients. Normalization keeps training stable.

In [ ]:
float_cols = [c for c in df.select_dtypes(include='float64').columns
              if c not in {'TransactionID', 'isFraud'}]

scaler = StandardScaler()
df[float_cols] = scaler.fit_transform(df[float_cols])

print(f'Normalized {len(float_cols)} float columns.')
print(f'TransactionAmt after scaling — mean: {df["TransactionAmt"].mean():.4f}, std: {df["TransactionAmt"].std():.4f}')

### 1.9 — Entity Column Identification

This is the most important step connecting Phase 01 to Phase 03.

The core idea of relational fraud detection: the **same card, device, or email** appears across multiple transactions. That shared connection becomes a graph edge. Below we list which columns represent these shared entities and how many unique values each has — that tells us how many nodes of each type we'll create.

In [ ]:
entity_cols = {
    'card1':         'Card Group A',
    'card2':         'Card Group B',
    'card3':         'Card Group C',
    'card4':         'Card Network (Visa/MC/etc.)',
    'card5':         'Card Group E',
    'card6':         'Card Type (debit/credit)',
    'addr1':         'Billing Zip',
    'addr2':         'Billing Country',
    'P_emaildomain': 'Purchaser Email Domain',
    'R_emaildomain': 'Recipient Email Domain',
    'DeviceType':    'Device Type',
    'DeviceInfo':    'Device Model',
}

print(f'{"Column":<20} {"Role":<30} {"Unique Values":>15}')
print('-' * 68)
for col, role in entity_cols.items():
    if col in df.columns:
        print(f'{col:<20} {role:<30} {df[col].nunique():>15,}')

### 1.10 — Validation & Save

Final sanity checks before moving on. We confirm no nulls, no object columns, label is intact, IDs are unique, and no infinity values slipped through normalization.

In [ ]:
checks = {
    'Zero remaining nulls':       df.isnull().sum().sum() == 0,
    'No object columns':          (df.dtypes == 'object').sum() == 0,
    'isFraud values are 0 and 1': set(df['isFraud'].unique()) == {0, 1},
    'TransactionID is unique':    df['TransactionID'].nunique() == len(df),
    'No inf values':              not np.isinf(df.select_dtypes(include=np.number).values).any(),
}

all_pass = True
for name, result in checks.items():
    status = '✓' if result else '✗'
    print(f'  {status}  {name}')
    if not result:
        all_pass = False

if all_pass:
    df.to_csv('/content/sample_data/df_preprocessed.csv', index=False)
    print(f'\nSaved → df_preprocessed.csv  ({df.shape[0]:,} rows x {df.shape[1]} cols)')
    print('Phase 01 complete. Next → Phase 02: Exploratory Data Analysis.')

---
## Phase 02 — Exploratory Data Analysis

>*Distribution analysis, fraud patterns by feature, transaction amount by fraud label, temporal patterns, correlation heatmap.*

---
## Phase 03 — Graph Construction

> Build heterogeneous graph from entity columns, define node types and edge types, construct adjacency structures, assign node feature matrices.*

---
## Phase 04 — GNN Model

> *Message passing from scratch in PyTorch, heterogeneous graph handling, attention mechanism (HGT or GAT-style).*

---
## Phase 05 — Training

> *Training loop, weighted cross-entropy / focal loss for class imbalance, optimizer, learning rate schedule, train/val/test split.*

---
## Phase 06 — Evaluation & Results

> *AUC-ROC, F1, precision-recall curve, confusion matrix, ablation studies, baseline comparison (MLP vs GNN), t-SNE embedding visualization.*